In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Exploratory Data Analysis

This notebook performs an exploratory data analysis on `supermarket_net_sales_forecast_prepared` produced by **Notebook 01 Data Preparation**.

**Scenario:** supermarket_net_sales_forecast  
**Generated:** 2026-06-03  

**Goals:**
1. Understand the shape, types, and basic statistics of the prepared dataset
2. Analyze the distribution of the target variable across stores
3. Detect trends, seasonality, and outliers
4. Examine correlations between features
5. Analyze features by group (media spend, weather, pricing, loyalty, etc.)
6. Identify collinear features to exclude from modeling
7. Compare store-level behavior to guide profiling and clustering

## Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

print("\u2705 Packages loaded")

## Configuration

In [ ]:
# CUSTOMIZED: Updated lakehouse and table names for supermarket_net_sales_forecast scenario
date_var = 'WEEK_START_DT'
unique_id = 'STORE_LOCATION_ID'
y = 'TOTAL_NET_SALES'
frequency = 'W'

LAKEHOUSE_NAME = "ts_mmm"
INPUT_TABLE = "supermarket_net_sales_forecast_prepared"

print(f"Target: {y}, ID: {unique_id}, Date: {date_var}, Freq: {frequency}")
print(f"Input: {LAKEHOUSE_NAME}.{INPUT_TABLE}")

## Load Data

In [ ]:
# CUSTOMIZED: Read from Lakehouse table directly (no prefix needed when session is attached)
df = spark.table(INPUT_TABLE).toPandas()

df[date_var] = pd.to_datetime(df[date_var])
df = df.sort_values([unique_id, date_var]).reset_index(drop=True)

print(f"\u2705 Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Stores: {df[unique_id].nunique()}, Date range: {df[date_var].min().date()} to {df[date_var].max().date()}")

# 1. Basic Overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes.value_counts().to_string())
print(f"\nMissing values per column:")
na_summary = df.isna().sum()
na_summary = na_summary[na_summary > 0]
if len(na_summary) > 0:
    print(na_summary.to_string())
else:
    print("  None")
print(f"\nDuplicates: {df.duplicated().sum()}")

In [ ]:
df.describe(include='all').T

In [ ]:
df.head(10)

# 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(df[y].dropna(), bins=50, edgecolor='white', alpha=0.8)
axes[0].set_title(f'Distribution of {y}')
axes[0].set_xlabel(y)
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Box plot by store
store_medians = df.groupby(unique_id)[y].median().sort_values()
ordered_ids = store_medians.index.tolist()
data_for_box = [df.loc[df[unique_id] == sid, y].dropna().values for sid in ordered_ids]
bp = axes[1].boxplot(data_for_box, vert=True, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
axes[1].set_title(f'{y} by Store (ordered by median)')
axes[1].set_xlabel('Store (rank)')
axes[1].set_ylabel(y)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Log-scale histogram
log_vals = np.log1p(df[y].dropna())
axes[2].hist(log_vals, bins=50, edgecolor='white', alpha=0.8, color='coral')
axes[2].set_title(f'Log(1 + {y})')
axes[2].set_xlabel(f'log(1 + {y})')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Skewness: {df[y].skew():.2f}, Kurtosis: {df[y].kurtosis():.2f}")
print(f"Zeros: {(df[y] == 0).sum()}, Negatives: {(df[y] < 0).sum()}, NaN: {df[y].isna().sum()}")

# 3. Time Series Trends

In [ ]:
# Aggregate across all stores
agg = df.groupby(date_var)[y].agg(['sum', 'mean', 'median', 'std']).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(agg[date_var], agg['sum'], linewidth=1.5)
axes[0].set_title(f'Total {y} per Week (All Stores)')
axes[0].set_ylabel(f'Total {y}')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[0].grid(True, alpha=0.3)

axes[1].plot(agg[date_var], agg['mean'], label='Mean', linewidth=1.2)
axes[1].fill_between(agg[date_var], agg['mean'] - agg['std'], agg['mean'] + agg['std'], alpha=0.2, label='\u00b1 1 Std')
axes[1].set_title(f'Mean {y} per Store per Week (\u00b1 1 Std)')
axes[1].set_ylabel(f'Mean {y}')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Individual store time series (sample of 9)
store_ids = df[unique_id].unique()
sample_n = min(9, len(store_ids))
np.random.seed(42)
sample_ids = np.random.choice(store_ids, sample_n, replace=False)

ncols = 3
nrows = (sample_n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows), sharex=True)
axes = axes.flatten()

for i, sid in enumerate(sorted(sample_ids)):
    ts = df[df[unique_id] == sid].sort_values(date_var)
    axes[i].plot(ts[date_var], ts[y], linewidth=1)
    axes[i].set_title(f'Store {sid}')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sample Store Time Series', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# 4. Seasonality Analysis

In [ ]:
df['_month'] = df[date_var].dt.month
df['_week'] = df[date_var].dt.isocalendar().week.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly seasonality
monthly = df.groupby('_month')[y].agg(['mean', 'std']).reset_index()
axes[0].bar(monthly['_month'], monthly['mean'], yerr=monthly['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_title(f'Monthly Seasonality (Mean {y} \u00b1 Std)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel(f'Mean {y}')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Weekly seasonality (average across all stores)
weekly = df.groupby('_week')[y].mean().reset_index()
axes[1].plot(weekly['_week'], weekly[y], marker='.', markersize=4, linewidth=1)
axes[1].set_title(f'Weekly Seasonality (Mean {y} by ISO Week)')
axes[1].set_xlabel('ISO Week')
axes[1].set_ylabel(f'Mean {y}')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print peak months/weeks
peak_month = monthly.loc[monthly['mean'].idxmax(), '_month']
low_month = monthly.loc[monthly['mean'].idxmin(), '_month']
peak_week = weekly.loc[weekly[y].idxmax(), '_week']
low_week = weekly.loc[weekly[y].idxmin(), '_week']
print(f"Peak month: {peak_month}, Low month: {low_month}")
print(f"Peak ISO week: {peak_week}, Low ISO week: {low_week}")
print(f"Monthly seasonal amplitude: {(monthly['mean'].max() - monthly['mean'].min()) / monthly['mean'].mean() * 100:.1f}%")

df.drop(columns=['_month', '_week'], inplace=True)

# 5. Store-Level Summary Statistics

In [ ]:
store_stats = df.groupby(unique_id)[y].agg(
    mean='mean', median='median', std='std',
    min='min', max='max', count='count',
).reset_index()
store_stats['zeros'] = df.groupby(unique_id)[y].apply(lambda x: (x == 0).sum()).values
store_stats['na_count'] = df.groupby(unique_id)[y].apply(lambda x: x.isna().sum()).values
store_stats['cv'] = store_stats['std'] / store_stats['mean']
store_stats['range'] = store_stats['max'] - store_stats['min']

print(f"Store-level summary ({len(store_stats)} stores):")
print(f"  Mean sales range: {store_stats['mean'].min():,.0f} to {store_stats['mean'].max():,.0f}")
print(f"  CV range: {store_stats['cv'].min():.3f} to {store_stats['cv'].max():.3f}")
print(f"  Stores with zeros: {(store_stats['zeros'] > 0).sum()}")
print(f"  Stores with NAs: {(store_stats['na_count'] > 0).sum()}")

store_stats.sort_values('mean', ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean vs CV scatter
axes[0].scatter(store_stats['mean'], store_stats['cv'], alpha=0.6, s=40)
axes[0].set_title('Store Mean vs Coefficient of Variation')
axes[0].set_xlabel(f'Mean {y}')
axes[0].set_ylabel('CV (std/mean)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
axes[0].axhline(y=store_stats['cv'].median(), color='r', linestyle='--', alpha=0.5, label=f'Median CV={store_stats["cv"].median():.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ranked mean sales
ranked = store_stats.sort_values('mean')
axes[1].barh(range(len(ranked)), ranked['mean'], color='steelblue', alpha=0.7)
axes[1].set_title(f'Stores Ranked by Mean {y}')
axes[1].set_xlabel(f'Mean {y}')
axes[1].set_ylabel('Store (rank)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

# 6. Correlation Analysis

In [ ]:
# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != unique_id]

if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    
    # Top correlations with target
    if y in corr.columns:
        target_corr = corr[y].drop(y).sort_values(key=abs, ascending=False)
        print(f"Top 20 correlations with {y}:")
        print(target_corr.head(20).to_string())
        print(f"\n... ({len(target_corr)} numeric features total)")
    
    # Heatmap (limit to top features for readability)
    top_n = min(25, len(numeric_cols))
    if y in corr.columns:
        top_cols = [y] + target_corr.head(top_n - 1).index.tolist()
    else:
        top_cols = numeric_cols[:top_n]
    
    corr_sub = corr.loc[top_cols, top_cols]
    
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(corr_sub.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(top_cols)))
    ax.set_yticks(range(len(top_cols)))
    ax.set_xticklabels(top_cols, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(top_cols, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f'Correlation Matrix (top {top_n} features by correlation with {y})')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")

# 7. Outlier Detection

In [ ]:
# IQR-based outlier detection per store
outlier_counts = []
for sid in df[unique_id].unique():
    vals = df.loc[df[unique_id] == sid, y].dropna()
    if len(vals) < 4:
        continue
    q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((vals < lower) | (vals > upper)).sum()
    outlier_counts.append({'store': sid, 'n_outliers': n_outliers, 'pct': n_outliers / len(vals) * 100})

outlier_df = pd.DataFrame(outlier_counts).sort_values('n_outliers', ascending=False)

print(f"Outlier summary (IQR method):")
print(f"  Stores with outliers: {(outlier_df['n_outliers'] > 0).sum()} / {len(outlier_df)}")
print(f"  Total outlier observations: {outlier_df['n_outliers'].sum()}")
print(f"  Max outliers in a single store: {outlier_df['n_outliers'].max()} ({outlier_df['pct'].max():.1f}%)")
print(f"\nTop 10 stores by outlier count:")
outlier_df.head(10)

# 8. Feature Distributions

In [ ]:
# Categorical feature value counts
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in [unique_id, date_var]]

if cat_cols:
    print(f"Categorical features ({len(cat_cols)}):")
    for col in cat_cols[:10]:
        n_unique = df[col].nunique()
        print(f"\n  {col} ({n_unique} unique):")
        if n_unique <= 20:
            print(df[col].value_counts().head(10).to_string())
        else:
            print(f"    (too many unique values to display)")
else:
    print("No categorical features found.")

In [ ]:
# Numeric feature histograms (exclude target and id)
plot_cols = [c for c in numeric_cols if c != y]
plot_cols = plot_cols[:16]  # Limit to 16 for readability

if plot_cols:
    ncols = 4
    nrows = (len(plot_cols) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
    axes = axes.flatten()

    for i, col in enumerate(plot_cols):
        vals = df[col].dropna()
        axes[i].hist(vals, bins=30, edgecolor='white', alpha=0.7)
        axes[i].set_title(col, fontsize=9)
        axes[i].set_ylabel('Count')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Numeric Feature Distributions (sample)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric features to plot (besides target).")

# 9. Feature Analysis

Features are grouped by domain category. Correlations with the target (TOTAL_NET_SALES) are shown for each numeric feature.

# \u2705 CHECK POINT with the data scientist: ask if BASELINE_WEEKLY_SALES, TOTAL_TRANSACTIONS and TOTAL_GROSS_MARGIN should be removed because they are collinear or if the Data Scientist wants to perform a PCA on those features.

In [ ]:
# Section 9: Feature Analysis - Group features by category
all_cols = df.columns.tolist()
exclude = [date_var, unique_id, y]

feature_groups = {
    'Store Characteristics': ['STORE_SELLING_AREA_SQFT', 'STORE_ARCHETYPE', 'REGION'],
    'Competition': [c for c in all_cols if 'COMPETITOR' in c or c in ['NEARBY_COMPETITOR_STORE_COUNT', 'HAS_COMPETITOR_IN_TRADE_AREA', 'IS_SOLE_STORE_IN_POSTAL_AREA', 'STORE_COUNT_IN_POSTAL_AREA']],
    'Media Spend': [c for c in all_cols if c.startswith('SPEND_')],
    'Volume / Impressions': [c for c in all_cols if c.startswith('VOLUME_')],
    'Pricing / Price Index': [c for c in all_cols if 'PRICE_INDEX' in c or c in ['DISCOUNT_FLYER', 'DISCOUNT_MEMBER_PRICING', 'AVG_COMPETITIVE_PRICE_INDEX']],
    'Weather': [c for c in all_cols if 'TEMPERATURE' in c or 'PRECIPITATION' in c or 'SNOWFALL' in c or 'DAYS_BELOW_ZERO' in c or 'DAYS_WITH_PRECIPITATION' in c],
    'Economic Indicators': [c for c in all_cols if 'CPI' in c or 'INFLATION' in c or 'UNEMPLOYMENT' in c],
    'Loyalty': [c for c in all_cols if 'LOYALTY' in c or c == 'SALES_ON_LOYALTY_CARD'],
    'Service Quality': [c for c in all_cols if 'SERVICE_GAP' in c or c == 'AVG_SERVICE_GAP_RATIO'],
    'Holidays': [c for c in all_cols if c.startswith('HOLIDAY_')],
    'Events': [c for c in all_cols if c.startswith('EVENT_')],
    'Market Share': [c for c in all_cols if 'MARKET_SHARE' in c],
    'Baseline / Transactions (COLLINEAR)': ['BASELINE_WEEKLY_SALES', 'TOTAL_TRANSACTIONS', 'TOTAL_GROSS_MARGIN'],
}

print("=" * 70)
print("FEATURE GROUP ANALYSIS")
print("=" * 70)
for group, cols in feature_groups.items():
    found = [c for c in cols if c in all_cols]
    icon = '\U0001f534' if 'COLLINEAR' in group else '\U0001f7e2'
    print(f"\n{icon} {group} ({len(found)} features):")
    for c in found:
        dtype = str(df[c].dtype) if c in df.columns else 'N/A'
        corr_val = target_corr.get(c, 'N/A') if c in numeric_cols else 'categorical'
        if isinstance(corr_val, float):
            print(f"    {c:45s} {dtype:12s} corr={corr_val:+.3f}")
        else:
            print(f"    {c:45s} {dtype:12s} {corr_val}")

# Count unassigned features
all_assigned = set()
for cols in feature_groups.values():
    all_assigned.update(cols)
unassigned = [c for c in all_cols if c not in all_assigned and c not in exclude]
if unassigned:
    print(f"\n\u26a0\ufe0f Unassigned features ({len(unassigned)}):")
    for c in unassigned:
        print(f"    {c}")
print("\n" + "=" * 70)

# 10. Summary

Key findings to carry into profiling and clustering (NB03-NB04):

In [ ]:
n_stores = df[unique_id].nunique()
n_weeks = df[date_var].nunique()
total_obs = len(df)
na_pct = df[y].isna().sum() / total_obs * 100
zero_pct = (df[y] == 0).sum() / total_obs * 100
cv_range = f"{store_stats['cv'].min():.3f} - {store_stats['cv'].max():.3f}"
mean_range = f"{store_stats['mean'].min():,.0f} - {store_stats['mean'].max():,.0f}"

print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)
print(f"Stores:              {n_stores}")
print(f"Weeks:               {n_weeks}")
print(f"Total observations:  {total_obs:,}")
print(f"Missing target (%):  {na_pct:.1f}%")
print(f"Zero target (%):     {zero_pct:.1f}%")
print(f"Mean sales range:    {mean_range}")
print(f"CV range:            {cv_range}")
print(f"Skewness:            {df[y].skew():.2f}")
print(f"Kurtosis:            {df[y].kurtosis():.2f}")
print(f"Numeric features:    {len(numeric_cols)}")
print(f"Categorical features:{len(cat_cols)}")
print(f"Outlier stores:      {(outlier_df['n_outliers'] > 0).sum()} / {n_stores}")
print(f"Total outliers:      {outlier_df['n_outliers'].sum()} obs")
print("=" * 60)